# CMT Patch-Based Inpainting Training

Train the CMT model with **patch-based training** for high-quality vessel inpainting.

---
⚠️ **Before running:** Runtime → Change runtime type → **T4 GPU** or **V100**

**Features:**
- Patch-based training (64×64 patches from full resolution)
- Enhanced metrics: PSNR, SSIM, Wasserstein, RMSE
- Automated checkpoint downloading
- Google Drive integration for results

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repository
!git clone https://github.com/C0d3Crush/arcade-xray-inpainting.git
%cd /content/arcade-xray-inpainting

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q scipy  # For vessel padding

## 2. Load ARCADE Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set ARCADE dataset path
ARCADE_ZIP = '/content/drive/MyDrive/CMT/arcade.zip'  # Update path as needed

# Extract ARCADE dataset
!unzip -q -o "$ARCADE_ZIP" -d /content/arcade-xray-inpainting/

# Verify dataset structure
!find data/arcade -name '*.json' | head -3
!ls -la data/arcade/syntax/

## 3. Optimize Training Setup

In [ ]:
# Cache masks and annotations for 10× faster training
!make cache-data

# Verify caching worked
!ls -la data/masks_cache/
print("✓ Data caching complete - training will be much faster!")

## 4. Patch-Based Training Configuration

**Training Strategy:**
- **Patch Mode**: Extract 64×64 patches from full-resolution images
- **Dataset Multiplication**: 16 patches per image → 16× more training data
- **Vessel Padding**: 3px context around vessels for better inpainting
- **High Epochs**: 100 epochs for production-quality model

In [ ]:
# Setup Google Drive checkpoint mirroring
!mkdir -p /content/drive/MyDrive/CMT/patch_checkpoints

# Train CMT with PATCH MODE for best quality
!python src/train.py \
    --train_img data/arcade/syntax/train/images \
    --train_ann data/arcade/syntax/train/annotations/train.json \
    --val_img data/arcade/syntax/val/images \
    --val_ann data/arcade/syntax/val/annotations/val.json \
    --train_mask data/masks_cache/train \
    --val_mask data/masks_cache/val \
    --patch_mode \
    --input_size 64 \
    --patches_per_image 16 \
    --epochs 100 \
    --batch_size 32 \
    --device cuda \
    --output_dir checkpoints \
    --save_every 10

## 5. Training Monitoring

In [ ]:
# Plot training metrics
!make plot

# Display the plot
from IPython.display import Image, display
display(Image('training_plot.png'))

# Show final metrics
import pandas as pd
if os.path.exists('checkpoints/training_log.csv'):
    df = pd.read_csv('checkpoints/training_log.csv')
    print("\n📊 Final Training Results:")
    print(f"Best Val PSNR: {df['val_psnr'].max():.2f} dB")
    print(f"Best Val SSIM: {df['val_ssim'].max():.4f}")
    print(f"Final Train Loss: {df['train_loss'].iloc[-1]:.6f}")

## 6. Test the Trained Model

In [ ]:
# Generate test samples and run inference
!make prepare-patch-samples
!make inference
!make training-comparison

# Display results
print("🎯 Patch-Based Inpainting Results:")
display(Image('outputs/samples/patch_training_comparison.png'))

## 7. Download Trained Model

In [ ]:
# Copy to Google Drive for backup
!cp checkpoints/best.pth /content/drive/MyDrive/CMT/patch_checkpoints/best_patch_v2.pth
!cp checkpoints/training_log.csv /content/drive/MyDrive/CMT/patch_checkpoints/training_log_patch_v2.csv
!cp training_plot.png /content/drive/MyDrive/CMT/patch_checkpoints/ 2>/dev/null || echo "Plot not found"

# Download to local machine
from google.colab import files
files.download('checkpoints/best.pth')
files.download('checkpoints/training_log.csv')

print("\n✅ Model Download Complete!")
print("📁 Backup saved to: /MyDrive/CMT/patch_checkpoints/")
print("💾 Downloaded to your local machine")

# Model info
import os
model_size = os.path.getsize('checkpoints/best.pth') / (1024*1024)
print(f"📊 Model size: {model_size:.1f} MB")

## 8. Advanced Training Options (Optional)

For even better results, try these configurations:

In [ ]:
# Option 1: Higher resolution patches (if you have good GPU)
# !python src/train.py --patch_mode --input_size 128 --patches_per_image 8 --epochs 100 --batch_size 16 --device cuda

# Option 2: More patches per image (longer training, better quality)
# !python src/train.py --patch_mode --input_size 64 --patches_per_image 32 --epochs 150 --batch_size 16 --device cuda

# Option 3: Resume training from checkpoint
# !python src/train.py --ckpt checkpoints/best.pth --epochs 150 --patch_mode --device cuda

print("💡 Advanced options available - uncomment to use!")

---

## Training Summary

**What this notebook does:**
1. ✅ **Patch-based training** - Extracts 64×64 patches from full-resolution images
2. ✅ **16× data multiplication** - 16 patches per image for better learning
3. ✅ **Vessel padding** - 3px context around vessels for realistic inpainting
4. ✅ **Enhanced metrics** - PSNR, SSIM, Wasserstein, RMSE tracking
5. ✅ **GPU acceleration** - Full CUDA training
6. ✅ **Automatic backup** - Google Drive checkpoint mirroring

**Expected improvements over smoke test:**
- PSNR: 30+ → 35-40+ dB
- Much smoother vessel inpainting
- Better anatomical structure preservation
- Production-ready model quality

**Use your new `best.pth`** in your local workflow for high-quality results!